In [1]:
%load_ext autoreload
%autoreload 2
from omegaconf import OmegaConf
from oat.data_module import factory
from oat.model_module.model import OAT
import torch

In [2]:
from torch.utils.data import DataLoader
from oat.data_module.wind_dataset import QueryBatch
data_config = OmegaConf.load('/scratch/mch/cdonner/observation_assimilation_transformer/configs/data/wind_data.yaml')
dataset = factory.create_dataset(data_config)
#train_dataloader, val_dataloader, test_dataloader = factory.create_dataset(data_config)
#batch = next(iter(train_dataloader))

loader = DataLoader(dataset, batch_size=16, shuffle=False)


/scratch/mch/cdonner/observation_assimilation_transformer/.venv/lib/python3.12/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [3]:
batch = next(iter(loader))

In [4]:
obs_data = batch['observation_data']
num_stations = obs_data['obs'].shape[1]
key_stations = 100
query = QueryBatch(
    coords=obs_data['coords'][:, key_stations:, :],
    topo=obs_data['topo'][:, key_stations:, :],
    geo=obs_data['geo'][:, key_stations:, :],
    obs=obs_data['obs'][:, key_stations:, :],
    mask_obs=obs_data['mask_obs'][:, key_stations:, :],
)
observation_data = {
    'coords': obs_data['coords'][:, :key_stations, :],
    'topo': obs_data['topo'][:, :key_stations, :],
    'geo': obs_data['geo'][:, :key_stations, :],
    'obs': obs_data['obs'][:, :key_stations, :],
    'mask_obs': obs_data['mask_obs'][:, :key_stations, :],
}
batch['observation_data'] = observation_data
batch['query_data'] = query

In [5]:
model = OAT()

In [6]:
with torch.no_grad():
    output = model(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'], )

In [ ]:

#with torch.no_grad():
# get gradients
pred, loss = model.forward_with_loss(global_data=batch['global_data'], observation_data=batch['observation_data'], query_data=batch['query_data'])
loss.backward()

In [23]:
for param in model.parameters():
    if param.grad is not None:
        print(param.grad.abs().sum())

tensor(3.3479)
tensor(0.0190)
tensor(0.0008)
tensor(0.0007)
tensor(3.2378e-05)
tensor(0.0352)
tensor(0.0016)
tensor(0.0328)
tensor(0.0028)
tensor(0.0005)
tensor(0.0006)
tensor(0.0377)
tensor(0.0018)
tensor(0.0778)
tensor(0.0029)
tensor(0.0008)
tensor(0.0007)
tensor(3.5533e-05)
tensor(0.0375)
tensor(0.0018)
tensor(0.0386)
tensor(0.0029)
tensor(0.0006)
tensor(0.0008)
tensor(0.0412)
tensor(0.0022)
tensor(0.0852)
tensor(0.0027)
tensor(6.5172)
tensor(0.0590)
tensor(0.0270)
tensor(0.0039)
tensor(0.0041)
tensor(0.0040)
tensor(0.3961)
tensor(0.0088)
tensor(0.3147)
tensor(0.0128)
tensor(0.0043)
tensor(0.0024)
tensor(0.6045)
tensor(0.0102)
tensor(1.1258)
tensor(0.0127)
tensor(0.0032)
tensor(0.0025)
tensor(0.0028)
tensor(0.2760)
tensor(0.0059)
tensor(0.3351)
tensor(0.0114)
tensor(0.0034)
tensor(0.0028)
tensor(0.5331)
tensor(0.0086)
tensor(0.9339)
tensor(0.0112)
tensor(6.5983)
tensor(0.0286)
tensor(0.0173)
tensor(0.0022)
tensor(0.0025)
tensor(0.0014)
tensor(0.3979)
tensor(0.0048)
tensor(0.4365)
te

In [29]:
output.shape

AttributeError: 'dict' object has no attribute 'shape'

In [36]:
torch.sum(batch['query_data']['obs'] == torch.nan)

tensor(0)

In [41]:
torch.sum(torch.any(pred['val_pred'] == torch.nan))

tensor(0)

In [42]:
# check if nans in output
torch.sum(output['val_pred'] == torch.nan)

tensor(0)

In [47]:
pred['val_pred'][2]

tensor([[-0.7943],
        [    nan],
        [-1.8782],
        [    nan],
        [-1.3191],
        [    nan],
        [    nan],
        [-0.7799],
        [    nan],
        [    nan],
        [    nan],
        [-1.4788],
        [-0.4745],
        [-1.1255],
        [-0.8918],
        [-0.8897],
        [-1.0826],
        [-1.2073],
        [-1.2287],
        [-1.1514],
        [-0.9346],
        [    nan],
        [-1.2657],
        [    nan],
        [-1.6528],
        [-1.0895],
        [-0.9621],
        [    nan],
        [-0.7659],
        [-1.6265],
        [-1.3637],
        [    nan],
        [-0.5671],
        [-2.0176],
        [    nan],
        [-0.6071],
        [    nan],
        [-1.9791],
        [-1.1748],
        [-0.9985],
        [    nan],
        [-1.6395],
        [-0.9241],
        [-1.1917],
        [-1.8473],
        [-1.4216],
        [-0.3853],
        [-0.2496],
        [-0.8603],
        [-0.4400],
        [-1.0012],
        [-1.2719],
        [-0.

In [49]:
batch['observation_data']['mask_obs'][2]

tensor([[ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True